# Configuration and Project Paths

In [1]:
from pathlib import Path
import csv
import hashlib
import html
import json
import re
from collections import Counter

import pandas as pd

In [2]:
RAW_PATH = Path("../../data/raw/Software.jsonl")

PROCESSED_DIR = Path("../../data/processed")
REPORTS_DIR = Path("../../reports")

CLEAN_OUTPUT_PATH = PROCESSED_DIR / "software_clean.jsonl"
REPORT_PATH = REPORTS_DIR / "02_data_preprocessing_report.md"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
CHUNK_SIZE = 5_000
MIN_REVIEW_ROWS = 10_000
MIN_REVIEW_TEXT_LENGTH = 3

In [4]:
assert RAW_PATH.exists(), f"Input file not found: {RAW_PATH.resolve()}"

print(f"Input : {RAW_PATH.resolve()}")
print(f"Output: {CLEAN_OUTPUT_PATH.resolve()}")

Input : E:\Project\my-project\FinalYearProject\ForeSightAI\data\raw\Software.jsonl
Output: E:\Project\my-project\FinalYearProject\ForeSightAI\data\processed\software_clean.jsonl


# Detect the input format

In [5]:
SUPPORTED_EXTENSIONS = {".json", ".jsonl", ".csv", ".xls", ".xlsx"}


In [6]:
def detect_file_format(path):
    extension = path.suffix.lower()

    if extension not in SUPPORTED_EXTENSIONS:
        raise ValueError(
            f"Unsupported file format: {extension}. "
            f"Supported formats: {sorted(SUPPORTED_EXTENSIONS)}"
        )

    return extension[1:]  # Remove the leading dot from the extension

In [7]:
file_format = detect_file_format(RAW_PATH)
print("Detected format:", file_format)

Detected format: jsonl


In [8]:
TEXT_CANDIDATES = [
    "text", "reviewtext", "review_text", "review_body", "reviewbody",
    "review", "body", "content", "comment"
]

RATING_CANDIDATES = ["rating", "overall", "stars", "score"]

REVIEW_LIKE_FIELDS = {
    "text", "reviewtext", "review_text", "review_body", "reviewbody",
    "review", "body", "content", "comment", "rating", "overall",
    "stars", "score", "user_id", "reviewerid", "reviewer_id"
}

METADATA_LIKE_FIELDS = {
    "title", "brand", "price", "description", "rank", "main_cat",
    "category", "categories", "image", "images"
}

def find_column(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    return None